##### Library imports

In [ ]:
import re, os, sys
import pandas as pd
import cv2 as cv
import SimpleITK as sitk
import numpy as np 
from skimage import measure, filters
import statsmodels.api as sm
import matplotlib.pyplot as plt
import json

##### Function definitions

In [ ]:
def window_stack_sitk(input_im, window_center=40, window_width=80):
    """
    Clamps values outside [min, max] to the edge values.
    
    Parameters
    ----------
    input_im : sitk.Image
        Input, unwindowed, brain image
    window_center : int, optional
        The center of the Hounsfield Unit (HU) scale in the windowed image. Default is 40 HU (brain).
    window_width : int, optional
        The total HU window width around the window_center, in the windowed image. Default is 80 HU (brain).

    Returns
    -------
    windowed_im : sitk.Image
        Windowed CT image with the desired tissue's visualization optimized (e.g., 0 - 80 HU for brain tissue)
    
    """
    img_min = window_center - (window_width // 2)
    img_max = window_center + (window_width // 2)


    windowed_im = sitk.IntensityWindowing(
        input_im, 
        windowMinimum=float(img_min), 
        windowMaximum=float(img_max), 
        outputMinimum=float(img_min), 
        outputMaximum=float(img_max)
    )
    
    return windowed_im

In [ ]:
def getLargestCC(blobs_labels):
    """Returns the largest connected component from an image containing blobs of discrete labels
    
    Parameters
    ----------
    blobs_labels : numpy.ndarray 
        A collection of discrete blobs (e.g., cerebrospinal fluid [CSF] spaces including the lateral ventricles, longitudinal fissure, sulci etc.)

    Returns
    -------
    largestCC: numpy.ndarray
        The largest connected component from a collection of discrete blobs (e.g. lateral ventricles from a collection of CSF spaces)  
    
    """
    # assume at least 1 CC apart from background
    if blobs_labels.max() == 0:
        raise ValueError('Blank segmentation, inspect processing up to here.')
    #this assumes that the background is the largest CC (label 0)
    largestCC = blobs_labels == np.argmax(np.bincount(blobs_labels.flat)[1:])+1 
    return largestCC

In [ ]:
def rotate_image(image, dimension = 3):
    """Rotates NIfTI coronal images so they are upright for visualization. Simple insights tool kit (SITK) utilizes the LPS coordinate system, 
       meaning image voxel coordinates are assumed to increase in the Right --> Left, Anterior --> Posterior, and Inferior --> Superior directions. 
       However, NIfTI images use the RAS coordinate system, which makes coronal slices (vertical cross-sections) appear inverted when read with SITK. 
       This function resamples the image such that it is visualized in upright direction, enabling interpretable visualization and processing. 
    
    Parameters
    ----------
    image : sitk.Image
        The original NIfTI image (Note that the direction cosine should be [-1,0,0,0,-1,0,0,0,1], although this is a coronal image, it is 
        assumed to be resampled and cropped such that the direction cosine corresponds to an axial volume)
    dimension : int, optional
        Dimensionality of the original and rotation-corrected image. Default is 3.
    
    Returns
    -------
    rotated_image : sitk.Image
        Rotation-corrected image, with a direction cosine of [1,0,0,0,1,0,0,0,1]
    """

    transform = sitk.AffineTransform(dimension)
    transform.SetCenter(image.TransformContinuousIndexToPhysicalPoint(np.array(image.GetSize())//2.0))
    s = image.GetSpacing()[2]

    matrix = np.array([[1.0,0.0,0.0],[0.0,1.0,0.0],[0.0,0.0,-1.0]])

   
    transform.SetMatrix(matrix.ravel())

    extreme_points = [image.TransformIndexToPhysicalPoint((0,0,0)), 
                      image.TransformIndexToPhysicalPoint((image.GetWidth(),0,0)),
                      image.TransformIndexToPhysicalPoint((image.GetWidth(),image.GetHeight(),0)),
                      image.TransformIndexToPhysicalPoint((0,image.GetHeight(),0)),
                      image.TransformIndexToPhysicalPoint((0,0,image.GetDepth())), 
                      image.TransformIndexToPhysicalPoint((image.GetWidth(),0,image.GetDepth())),
                      image.TransformIndexToPhysicalPoint((image.GetWidth(),image.GetHeight(),image.GetDepth())),
                      image.TransformIndexToPhysicalPoint((0,image.GetHeight(),image.GetDepth()))]

    inv_transform = transform.GetInverse()

    extreme_points_transformed = [inv_transform.TransformPoint(pnt) for pnt in extreme_points]
    min_x = min(extreme_points_transformed)[0]
    min_y = min(extreme_points_transformed, key=lambda p: p[1])[1]
    min_z = min(extreme_points_transformed, key=lambda p: p[2])[2]
    max_x = max(extreme_points_transformed)[0]
    max_y = max(extreme_points_transformed, key=lambda p: p[1])[1]
    max_z = max(extreme_points_transformed, key=lambda p: p[2])[2]

    # Use the original spacing (arbitrary decision).
    output_spacing = image.GetSpacing()
    # Identity cosine matrix.   
    output_direction = [1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0]
    # Minimal x,y coordinates are the new origin.
    output_origin = [min_x, min_y, min_z]
    
    # Compute grid size based on the physical size and spacing.
    output_size = image.GetSize()
    
    rotated_image = sitk.Resample(image, output_size, transform, sitk.sitkLinear, output_origin, output_spacing,
                                  output_direction)
    return rotated_image

In [ ]:
def segment_ventricles(cor_img_path, pc, coronal_plane_processing_params, visualize = False):

    """
    Processes the coronal CT plane at the level of the Posterior Commissure (PC) 
    and returns the segmented lateral ventricular and full brain masks for further analysis.

    This function isolates the specific coronal slice corresponding to the PC, 
    applies Gaussian blurring, and uses both Otsu and adaptive thresholding to 
    segment the brain tissue. It fills holes using flood-fill and morphological 
    closing, suppresses isolated noisy blobs, and breaks occlusions caused by 
    the choroid plexus using morphological opening. Finally, it determines the 
    ventricular tissue by subtracting the segmented brain tissue from the full brain mask. 

    Parameters
    ----------
    cor_img_path : str
        The file path to the aligned Coronal NIfTI volume.
    pc : tuple or list
        The (x, y, z) physical coordinates of the Posterior Commissure.
    coronal_plane_processing_params : dict
        A dictionary containing processing parameters including blur sizes, 
        kernel sizes, iterations, and threshold values.
    visualize : bool, optional
        If True, displays intermediate processing steps via matplotlib (default is False).

    Returns
    -------
    ventricles: numpy.ndarray
        Segmented lateral ventricles. 
    full_brain_mask : numpy.ndarray
        The solid, filled mask representing the entire brain area in the slice.
    coronal_plane: numpy.ndarray
        Brain windowed coronal slice at the level of the PC
    cor_pc_index: tuple
        Coronal image coordinates of the PC
    """

    #unpack processing parameters
    coronal_initial_gauss_blur = coronal_plane_processing_params["coronal_initial_gauss_blur"]
    coronal_close_kernel_size = coronal_plane_processing_params["coronal_close_kernel_size"]
    coronal_close_num_iterations = coronal_plane_processing_params["coronal_close_num_iterations"]
    coronal_adaptive_thresh_size = coronal_plane_processing_params["coronal_adaptive_thresh_size"]
    coronal_adaptive_thresh_mean_thresh = coronal_plane_processing_params["coronal_adaptive_thresh_mean_thresh"]
    suppress_label_thresh = coronal_plane_processing_params["suppress_label_thresh"]

    #Read in coronal plane
    cor_img = sitk.ReadImage(cor_img_path) 
    cor_img = rotate_image(cor_img)
    cor_img_arr = sitk.GetArrayFromImage(cor_img)

    #Window brain tissue
    cor_img_brain = window_stack_sitk(cor_img)   
    cor_img_brain_arr = sitk.GetArrayFromImage(cor_img_brain)

    #pick the coronal plane at the level of the PC
    cor_pc_index = cor_img_brain.TransformPhysicalPointToContinuousIndex(pc)
    pl  = int(np.round(cor_pc_index[1]))
    coronal_plane = cor_img_brain_arr[:,pl,:]
    cor_pl_org = cor_img_arr[:,pl,:]

    #Gaussian blurring for thresholding/segmentation (de-noising step)    
    coronal_gauss_blurred = cv.GaussianBlur(np.uint8(coronal_plane),(coronal_initial_gauss_blur,coronal_initial_gauss_blur), cv.BORDER_DEFAULT) 

    otsu_thresh = filters.threshold_otsu(coronal_gauss_blurred)
    coronal_bin = coronal_gauss_blurred > otsu_thresh    
    close_kernel = cv.getStructuringElement(cv.MORPH_ELLIPSE, (coronal_close_kernel_size,coronal_close_kernel_size))
    closed_coronal_brain_mask = cv.morphologyEx(np.uint8(coronal_bin), 
                                                cv.MORPH_CLOSE, 
                                                close_kernel, 
                                               iterations = coronal_close_num_iterations) 
    im_th = coronal_bin.copy()
    h, w = im_th.shape[:2]
    mask = np.zeros((h+2, w+2), np.uint8)
    im_floodfill = im_th.copy()
    # Floodfill from point (0, 0)
    res = cv.floodFill(np.uint8(im_floodfill), mask, (0,0), 255)
    full_brain_mask = (1-res[2]).copy()
    full_brain_mask = full_brain_mask[1:h+1,1:w+1]
    M = cv.moments(np.uint8(full_brain_mask))

    if M["m00"] != 0:
        bcX = int(M["m10"] / M["m00"])
        bcY = int(M["m01"] / M["m00"])
    else:
        bcX, bcY = full_brain_mask.shape[1]//2, full_brain_mask.shape[0]//2

    coronal_adapt_thresh = cv.adaptiveThreshold(coronal_gauss_blurred,1,cv.ADAPTIVE_THRESH_MEAN_C,
                                cv.THRESH_BINARY, coronal_adaptive_thresh_size, coronal_adaptive_thresh_mean_thresh)

    coronal_adapt_thresh_bg_supressed = np.where((1-full_brain_mask) > 0, 0 ,coronal_adapt_thresh)
    coronal_brain_mask = np.where(coronal_adapt_thresh_bg_supressed > 0, 1, 0)


    blobs_before_processing = measure.label(full_brain_mask - coronal_brain_mask, background = 0)
    
    #if there are small dark blobs in the otsu or adaptive thresholded brain mask, they stand out as independent
    #areas when subtracted from the full brain mask. So identifying them and labeling them as 1 before the 
    #ventricles are obtained will supress some of the spurious blobs
    try:
        coronal_brain_mask_small_holes_closed = coronal_brain_mask.copy()
        coronal_bin_closed = coronal_bin.copy()
        blob_sums = np.unique(blobs_before_processing, return_counts = True)[1]
        suppress_labels = np.where(blob_sums/(blob_sums[np.argsort(blob_sums)[::-1][1]]) < suppress_label_thresh)[0]
        for sl in suppress_labels:
            coronal_brain_mask_small_holes_closed = np.where(blobs_before_processing==sl,1,
                                                             coronal_brain_mask_small_holes_closed)
            coronal_bin_closed = np.where(blobs_before_processing==sl,1,coronal_bin_closed)
            
    except:
        coronal_brain_mask_small_holes_closed = coronal_brain_mask
        coronal_bin_closed = coronal_bin


    #To remove occlusions in ventricles by choroid plexus (cp)
    cp_break_kernel = np.array([[1,1,1],[0,1,0],[1,1,1]],'uint8')
    open_coronal_brain_mask = cv.morphologyEx(np.uint8(coronal_brain_mask_small_holes_closed), 
                                                        cv.MORPH_OPEN,
                                                        cp_break_kernel, 
                                                        iterations = coronal_close_num_iterations) 

    ventricle_mask_from_adaptive_0 = np.where(full_brain_mask-open_coronal_brain_mask>0,1,0)
    ventricle_mask_from_otsu_0 = np.where((1-coronal_bin)*full_brain_mask>0,1,0)
    ventricles = np.logical_or(ventricle_mask_from_otsu_0, ventricle_mask_from_adaptive_0)

    
    if visualize:
        plt.imshow(coronal_plane, cmap = 'gray')
        plt.title('Coronal plane')
        plt.show()
        plt.imshow(coronal_gauss_blurred,cmap = 'gray')
        plt.title('Coronal gauss blurred')
        plt.show()
        plt.imshow(coronal_bin,cmap = 'gray')
        plt.title('Coronal OTSU')
        plt.show()
        plt.imshow(full_brain_mask,cmap = 'gray')
        plt.title('Full brain mask')
        plt.show()
        plt.imshow(coronal_brain_mask,cmap = 'gray')
        plt.title('Adaptive coronal brain mask')
        plt.show()        
        plt.imshow(coronal_brain_mask_small_holes_closed, cmap = 'gray')
        plt.title('After supress labels - adaptive')
        plt.show()
        plt.imshow(coronal_bin_closed, cmap = 'gray')
        plt.title('After supress labels - otsu')
        plt.show()        
        plt.imshow(open_coronal_brain_mask,cmap = 'gray')
        plt.title('Adaptive coronal brain mask - Open')
        plt.show()
        plt.imshow(ventricles, cmap = 'gray')
        plt.title('Segmented lateral ventricles')
        plt.show()
        
    return ventricles, full_brain_mask, cor_pc_index, coronal_plane

In [ ]:
def process_venticles(ventricles, cor_pc_index, full_brain_mask, ventricles_processing_smoothing_params, visualize = False):

    """
    Processes and isolates the lateral ventricles in the coronal CT plane above the Posterior Commissure (PC).

    This function takes the initially processed ventricles, the PC index, and the 
    full brain mask to isolate the ventricular ROI above the PC. It cleans up 
    non-ventricular tissue using connected component analysis and dynamic ROI 
    cropping based on percentile heights. Finally, it dynamically smooths the 
    tissue based on a size criterion—adjusting the smoothing factor until the 
    ventricles represent a sufficient proportion of the full brain mask.

    Parameters
    ----------
    ventricles : numpy.ndarray
        The initially processed binary mask of the ventricles.
    cor_pc_index : tuple or list
        The (x, y, z) index coordinates of the Posterior Commissure.
    full_brain_mask : numpy.ndarray
        The binary mask representing the full CT brain volume in the current slice.
    ventricles_processing_smoothing_params : dict
        A dictionary containing processing parameters (ROI sizes, smoothing parameters etc.)
    visualize : bool, optional
        If True, displays intermediate processing steps via matplotlib (default is False).

    Returns
    -------
    ventricles_smoothed : numpy.ndarray
        The final, cleaned, and dynamically smoothed ventricular mask above the PC.
    """

    middle_roi_upper_dec = ventricles_processing_smoothing_params["middle_roi_upper_dec"]
    middle_roi_lower_inc = ventricles_processing_smoothing_params["middle_roi_lower_inc"]
    middle_roi_left_dec = ventricles_processing_smoothing_params["middle_roi_left_dec"]
    middle_roi_right_dec = ventricles_processing_smoothing_params["middle_roi_right_dec"]
    scale_upper_thresh = ventricles_processing_smoothing_params["scale_upper_thresh"]
    smoothing_factor = ventricles_processing_smoothing_params["smoothing_factor"]
    dec_thresh = ventricles_processing_smoothing_params["dec_thresh"]
    ventricle_to_brain_ratio_thresh = ventricles_processing_smoothing_params["ventricle_to_brain_ratio_thresh"]
                
    pt = int(ventricles.shape[0]-cor_pc_index[2])

    #Adding this mask before smoothing and connected component analysis as it does not exaggerate any split
    #between the LV's up and bottom part
    above_pc_mask = np.zeros(ventricles.shape)
    above_pc_mask[:pt,:] = 1
    ventricles_above_pc = np.where(ventricles*above_pc_mask>0,1,0)
 
    #to get the ventricles as one CC and cleanup the surroundings
    M = cv.moments(np.uint8(full_brain_mask))#ventricles_above_pc
    if M["m00"] != 0:
        bcX = int(M["m10"] / M["m00"])
        bcY = int(M["m01"] / M["m00"])
    else:
        bcX, bcY = full_brain_mask.shape[1]//2, full_brain_mask.shape[0]//2
    
    middle_roi = np.zeros(ventricles_above_pc.shape)
    middle_roi[bcY-middle_roi_upper_dec : bcY+middle_roi_lower_inc, 
               bcX-middle_roi_left_dec  : bcX+middle_roi_right_dec] = 1 
    
    clean_ven_1 = np.logical_or(middle_roi, ventricles_above_pc) 

    blobs_labels = measure.label(clean_ven_1, background = 0)
    largest_cc_ven = getLargestCC(blobs_labels)      
    [row_v, col_v] = (1-largest_cc_ven).nonzero()
    ventricles_clean_above_pc = ventricles_above_pc.copy()
    ventricles_clean_above_pc[row_v, col_v] = 0
    
    [vr,vc] = ventricles_clean_above_pc.nonzero()
    if len(vr) == 0:
        print("Warning: Ventricles mask is empty above the PC. Returning crude LV mask above PC")
        return ventricles_clean_above_pc
        
    lv_roi = np.zeros(ventricles_clean_above_pc.shape)
    iqr_row = np.percentile(vr,75) - np.percentile(vr,25)
    
    upper_thresh_row = int(np.round((np.percentile(vr,75) + scale_upper_thresh*iqr_row)))
    lv_roi[:upper_thresh_row,:] = 1  
    ventricles_clean_above_pc_1 = np.uint8(lv_roi*ventricles_clean_above_pc)

    if visualize:
        plt.imshow(ventricles,cmap = 'gray')
        plt.imshow(middle_roi, alpha = 0.2)
        plt.title('Ventricles with middle ROI shown')
        plt.show()
        plt.imshow(ventricles_clean_above_pc, cmap = 'gray')
        plt.title('Ventricles cleaned above PC')
        plt.show()
        plt.imshow(ventricles_clean_above_pc_1, cmap = 'gray')
        plt.title('Ventricles cleaned above PC, only until the 75th p')
        plt.show()

    #Dynamic smoothing based on what percentage of ventricles are left behind
    while True:
        ventricles_smoothed = cv.medianBlur(np.uint8(ventricles_clean_above_pc_1),smoothing_factor - dec_thresh)
        if visualize:            
            plt.imshow(ventricles_smoothed,cmap = 'gray')
            plt.title(f'Ventricles smoothed at {smoothing_factor - dec_thresh}')
            plt.show()
        
        if np.sum(ventricles_smoothed)/np.sum(full_brain_mask) >= ventricle_to_brain_ratio_thresh: 
            break
        else:
            print('Smoothing factor being reduced')
            dec_thresh = dec_thresh + 2
            if dec_thresh > 4:
                print('ventricles too small')
                break
    if visualize:                    
        plt.imshow(ventricles_smoothed,cmap = 'gray')
        plt.title('Final ventricles smoothed')
        plt.show()
                
    return ventricles_smoothed

In [ ]:
def num_connected_comps_cv(clean_ventricles, visualize=False): 
    """
    Evaluates the cleaned ventricular CT mask to determine if the left and right 
    lateral ventricles are structurally connected or disconnected.

    This function performs connected component analysis to isolate the largest 
    ventricular blobs. It evaluates spatial characteristics—such as distance 
    from the image center, vertical alignment, and relative blob areas—using 
    domain knowledge to classify the ventricles as a disconnected case or not. 
    It also cleans up spurious non-ventricular tissue based on these checks.

    Parameters
    ----------
    clean_ventricles : numpy.ndarray
        The binary mask of the ventricles after initial processing and cleaning.
    visualize : bool, optional
        If True, displays the final processed blobs via matplotlib (default is False).

    Returns
    -------
    disconnected_case : bool
        True if the left and right lateral ventricles appear structurally 
        disconnected based on spatial checks; otherwise False.
    processed_blobs : numpy.ndarray
        The updated binary mask containing only the validated ventricular blobs.
    """

    disconnectedLV_x_diff_tol = 25
    area_check_thresh = 0.9
    
    blobs_labels = measure.label(clean_ventricles, background=0)

    if len(np.unique(blobs_labels)) <= 1:
        print("Warning: No distinct connected components found in the mask.")
        return False, clean_ventricles

    processed_blobs = blobs_labels

    # If there are more than 3 disconnected nb components, keep the largest 3
    if len(np.unique(blobs_labels)) > 3: # More than background and 2 more blobs
        top_3_args = np.argsort(np.bincount(blobs_labels.flatten())[1:])[::-1][:3]
        processed_blobs_1 = np.where(blobs_labels == top_3_args[0]+1, blobs_labels, 0)
        [pr1, pc1] = processed_blobs_1.nonzero()
        processed_blobs_2 = np.where(blobs_labels == top_3_args[1]+1, blobs_labels, 0)
        [pr2, pc2] = processed_blobs_2.nonzero()
        processed_blobs_3 = np.where(blobs_labels == top_3_args[2]+1, blobs_labels, 0)
        [pr3, pc3] = processed_blobs_3.nonzero()
 
        if len(pr1) > 0 and len(pr2) > 0 and len(pr3) > 0:
            closest_inds = np.argmin(np.array([
                np.abs(np.median(pr1) - np.median(pr2)), 
                np.abs(np.median(pr1) - np.median(pr3)),
                np.abs(np.median(pr2) - np.median(pr3))
            ]))
            
            if closest_inds == 0:
                processed_blobs = processed_blobs_1 + processed_blobs_2 # 1 and 2
            elif closest_inds == 1:
                processed_blobs = processed_blobs_1 + processed_blobs_3 # 1 and 3
            else:
                processed_blobs = processed_blobs_2 + processed_blobs_3
        else:
            processed_blobs = processed_blobs_1 + processed_blobs_2

    disconnected_case = False

    # At this point, we should have background, plus <= 2 blobs
    rem_labels = np.unique(processed_blobs)
    new_labels = range(len(rem_labels))
    
    
    map_dict = {r: n for r, n in zip(rem_labels, new_labels)}
    new_blobs = np.zeros_like(processed_blobs)
    
    for k, v in map_dict.items():
        new_blobs[processed_blobs == k] = v
        
    processed_blobs = new_blobs
    
    [pr, pc] = processed_blobs.nonzero()
    
    
    if len(pc) == 0:
        return False, np.where(processed_blobs > 0, 1, 0)
        
    im_cen = (max(pc) + min(pc)) // 2
    
    if len(np.unique(processed_blobs)) == 3: # 3 including background
        # Note: label 0 is background. Argsort of bincount sorts by area ascending.
        counts = np.bincount(processed_blobs.flatten())
        
        smallest_blob_arg = np.argsort(counts)[0]
        smallest_blob = np.where(processed_blobs == smallest_blob_arg, 1, 0)
        [sb_r, sb_c] = smallest_blob.nonzero()
        smallest_blob_height = (min(sb_r) + max(sb_r)) // 2
        smallest_blob_cen_x = (min(sb_c) + max(sb_c)) // 2

        mid_blob_arg = np.argsort(counts)[1]
        mid_blob = np.where(processed_blobs == mid_blob_arg, 1, 0)
        [mb_r, mb_c] = mid_blob.nonzero()
        mid_blob_height = (min(mb_r) + max(mb_r)) // 2
        mid_blob_cen_x = (min(mb_c) + max(mb_c)) // 2

        nv_mass = (smallest_blob_height < min(mb_r)) | (smallest_blob_height > max(mb_r)) # non-ventricular mass

        area_check = np.sum(mid_blob) - np.sum(smallest_blob) > area_check_thresh * np.sum(mid_blob)
        
        if ((np.abs(smallest_blob_cen_x - im_cen) <= disconnectedLV_x_diff_tol) & 
            (np.abs(mid_blob_cen_x - im_cen) <= disconnectedLV_x_diff_tol) &
            (~nv_mass) & (~area_check)):
            disconnected_case = True
        else: 
            processed_blobs = np.where(processed_blobs == smallest_blob_arg, 0, processed_blobs)
            
    processed_blobs = np.where(processed_blobs > 0, 1, 0) 

    if visualize:
        plt.imshow(processed_blobs, cmap='gray')
        plt.title('Output of connected component analysis')
        plt.show()
    
    return disconnected_case, processed_blobs

In [ ]:
def check_if_disconnected_case_too_small(lat_ventricles, full_brain_mask, disconnected_case, 
                                         process_left, process_right, 
                                         visualize=False):
    """
    Evaluates segmented CT lateral ventricles to correct falsely flagged disconnected cases and suppress noisy non-ventricular masses, OR
    suppress very small ventricles in truly disconnected cases.

    This function processes cases where the ventricles appear disconnected. It identifies 
    situations where one ventricle is suspiciously small or where a connected ventricle 
    has a small non-ventricular mass nearby that fooled the connected component analysis. 
    It updates the disconnected case flag, isolates the left and right ventricular contours, 
    and sets the 'process_left' or 'process_right' flags so the downstream pipeline handles 
    the corrected anatomy properly.

    Parameters
    ----------
    lat_ventricles : numpy.ndarray
        The binary mask of the segmented lateral ventricles.
    full_brain_mask : numpy.ndarray
        The binary mask of the entire brain volume in the current slice.
    disconnected_case : bool
        The initial flag indicating if the ventricles were deemed disconnected.
    process_left : int or bool
        Flag indicating whether the left ventricle should be processed downstream.
    process_right : int or bool
        Flag indicating whether the right ventricle should be processed downstream.       
    visualize : bool, optional
        If True, displays intermediate processing steps via matplotlib (default is False).

    Returns
    -------
    lat_ventricles : numpy.ndarray
        The updated lateral ventricles mask, potentially with small noisy blobs removed.
    disconnected_case : bool
        The updated flag, switched to False if a false-disconnect was identified.
    left_ven_con : numpy.ndarray
        The isolated contour/mask of the left ventricle.
    right_ven_con : numpy.ndarray
        The isolated contour/mask of the right ventricle.
    process_left : int or bool
        Updated flag for downstream processing of the left ventricle.
    process_right : int or bool
        Updated flag for downstream processing of the right ventricle.
    """
    lv_to_full_brain_thresh=0.01 #The ratio threshold of ventricle area to brain area to trigger the small-mass check
    connected_case_blob_ratio=0.65 #The allowable area difference ratio between two blobs to remain classified as disconnected 

    blob_labels = measure.label(lat_ventricles, background=0)
    [vr, vc] = lat_ventricles.nonzero()
    
    if len(vc) == 0:
        print("Warning: lat_ventricles mask is empty.")
        return lat_ventricles, disconnected_case, lat_ventricles.copy(), lat_ventricles.copy(), process_left, process_right

    vc_mid = (min(vc) + max(vc)) // 2
    
    left_ven_con = lat_ventricles.copy()
    left_ven_con[:, vc_mid:] = 0
    right_ven_con = lat_ventricles.copy()
    right_ven_con[:, :vc_mid] = 0

    # Do not process one LV if the entire mass is too small 
    if (len(np.unique(blob_labels)) > 2) and (np.sum(lat_ventricles) / np.sum(full_brain_mask) < lv_to_full_brain_thresh): 
        # Disconnected case, 2 LV - one might be small; 
        # Or connected case, small nv mass nearby when connected component analysis fails and falsely calls it disconnected

        counts = np.bincount(blob_labels.flatten())
        counts[0] = 0 # Zero out background count
        largest_2_indices = np.argsort(counts)[::-1][:2]

        larger_blob = np.where(blob_labels == largest_2_indices[0], 1, 0)
        [lbr, lbc] = larger_blob.nonzero()
        lbc_m = (min(lbc) + max(lbc)) // 2 if len(lbc) > 0 else 0

        smaller_blob = np.where(blob_labels == largest_2_indices[1], 1, 0)
        [sbr, sbc] = smaller_blob.nonzero()
        sbc_m = (min(sbc) + max(sbc)) // 2 if len(sbc) > 0 else 0

        blob_sums = [np.sum(smaller_blob), np.sum(larger_blob)]
        
        if blob_sums[1] > 0 and np.abs((blob_sums[1] - blob_sums[0]) / blob_sums[1]) > connected_case_blob_ratio:
            lat_ventricles = np.where(blob_labels == largest_2_indices[0], 1, 0)
            disconnected_case = False

        if sbc_m < lbc_m:
            process_left = 0 
        else:
            process_right = 0

        if visualize:
            plt.imshow(lat_ventricles, cmap='gray')
            plt.title('Disconnected case now processed as only one blob')
            plt.show()
            
    return lat_ventricles, disconnected_case, left_ven_con, right_ven_con, process_left, process_right

In [ ]:
def trace_contour(ven_contour, left_ven_con, right_ven_con, process_left, process_right, disconnected_case, cX, cY):
    """
    Traces the outer boundaries of the left and right CT ventricular contours.

    This function dynamically sets starting points and loop termination 
    conditions based on whether the ventricles are connected or disconnected. It 
    then performs a pixel-by-pixel directional trace along the contour boundaries. 
    Safety bounds are included to prevent infinite loops and out-of-bounds errors. 
    Finally, it filters out the inner 20% of points along the x-axis.

    Parameters
    ----------
    ven_contour : numpy.ndarray
        The binary image contour of the complete ventricular structure.
    left_ven_con : numpy.ndarray
        The isolated contour of the left ventricle.
    right_ven_con : numpy.ndarray
        The isolated contour of the right ventricle.
    process_left : int or bool
        Flag indicating if the left ventricle should be traced (1 for yes).
    process_right : int or bool
        Flag indicating if the right ventricle should be traced (1 for yes).
    disconnected_case : bool
        Flag indicating if the left and right ventricles are physically separated.
    cX : int
        The x-coordinate (column) serving as the horizontal center or starting reference.
    cY : int
        The y-coordinate (row) serving as the vertical top or starting reference.

    Returns
    -------
    left_con_cand_pts : list of tuples
        Filtered candidate (x, y) coordinates for the left ventricular boundary.
    right_con_cand_pts : list of tuples
        Filtered candidate (x, y) coordinates for the right ventricular boundary.
    left_pts : list of tuples
        The complete set of traced (x, y) coordinates for the left ventricle.
    right_pts : list of tuples
        The complete set of traced (x, y) coordinates for the right ventricle.
    """
    [vc_r, vc_c] = ven_contour.nonzero()
    if len(vc_c) == 0:
        return [], [], [], []
        
    midpoint_c = int((min(vc_c) + max(vc_c)) // 2)
    max_iters = 10000 # Safety limit to prevent infinite loops
    max_row, max_col = ven_contour.shape
    
    left_pts = []
    right_pts = []

    # ==========================================
    # Unified Processing: Left Ventricle
    # ==========================================
    if process_left == 1:
        # 1. Setup variables based on disconnected case type
        if not disconnected_case:
            [lvc_r, lvc_c] = ven_contour[:, :midpoint_c].nonzero()
            mask_to_check = ven_contour
            
            try: 
                lv_tp_x = min(lvc_c[lvc_r == cY])
            except: 
                lv_tp_x = min(lvc_c) if len(lvc_c) > 0 else cY
            
            # Dynamic stop condition to be negated for the connected case
            stop_condition = lambda r, c: (c < cX) and (r <= cY)
        else:
            [lvc_r, lvc_c] = left_ven_con.nonzero()
            mask_to_check = left_ven_con
            
            try: 
                lv_tp_x = min(lvc_c[lvc_r == cY])
            except: 
                lv_tp_x = min(lvc_c) if len(lvc_c) > 0 else cY
            
            stop_row_l, stop_col_l = max(lvc_r[lvc_c == max(lvc_c)]), max(lvc_c) - 2 #stopping a bit early to avoid midline atrophy
            
            # Dynamic stop condition to be negated for the disconnected case
            stop_condition = lambda r, c: (r <= cY) and not (r == stop_row_l and c == stop_col_l)
            
        # Initialize Starting Coordinates
        row, col = cY, lv_tp_x
        if mask_to_check[row, col] == 0 and len(lvc_r[lvc_c == col]) > 0:
            row = max(lvc_r[lvc_c == col])

        # 2. Execute Tracing Loop
        north_west_count = 0
        south_step = 0
        iters = 0
        
        while stop_condition(row, col):
            # SAFETY CHECKS: Prevent out-of-bounds and infinite loops
            if iters > max_iters or row < 1 or row >= max_row - 1 or col < 1 or col >= max_col - 1:
                break
            iters += 1
            
            if (ven_contour[row-1, col-1] > 0) & (north_west_count <= 10): # north-west
                left_pts.append((col-1, row-1))
                row, col = row - 1, col - 1
                north_west_count += 1
                south_step = 0
            elif (ven_contour[row-1, col] > 0) & (south_step == 0): # north
                left_pts.append((col, row-1))
                row, col = row - 1, col
                south_step = 0
            elif (ven_contour[row-1, col+1] > 0): # north-east
                left_pts.append((col+1, row-1))
                row, col = row-1, col+1     
                south_step = 0
            elif (ven_contour[row, col+1] > 0): # east
                left_pts.append((col+1, row))
                row, col = row, col+1 
                south_step = 0
            elif (ven_contour[row+1, col+1] > 0): # south-east
                left_pts.append((col+1, row+1))
                row, col = row+1, col+1
                south_step = 0
            else: # south
                left_pts.append((col, row+1)) 
                row, col = row+1, col     
                south_step = 1
    
    # ==========================================
    # Unified Processing: Right Ventricle
    # ==========================================
    if process_right == 1:
        # 1. Setup variables based on disconnected case type
        if not disconnected_case:
            [rvc_r, rvc_c] = ven_contour[:, midpoint_c + 1:].nonzero()
            mask_to_check = ven_contour
            
            try: 
                rv_tp_x = max(rvc_c[rvc_r == cY]) + midpoint_c
            except: 
                rv_tp_x = max(rvc_c) + midpoint_c if len(rvc_c) > 0 else cY
            
            # Dynamic stop condition to be negated for the connected case
            stop_condition = lambda r, c: (c > cX) and (r <= cY)
            
            row, col = cY, rv_tp_x
            if mask_to_check[row, col] == 0 and len(rvc_r[rvc_c == max(rvc_c)]) > 0:
                row = max(rvc_r[rvc_c == max(rvc_c)])
                
        else:
            [rvc_r, rvc_c] = right_ven_con.nonzero()
            mask_to_check = right_ven_con
            
            try: rv_tp_x = max(rvc_c[rvc_r == cY])
            except: rv_tp_x = max(rvc_c) if len(rvc_c) > 0 else cY
            
            row, col = cY, rv_tp_x
            if mask_to_check[row, col] == 0 and len(rvc_r[rvc_c == col]) > 0:
                row = max(rvc_r[rvc_c == col])
                
            stop_row_r, stop_col_r = max(rvc_r[rvc_c == min(rvc_c)]), min(rvc_c) + 2 #starting a bit later to avoid midline atrophy
            
            # Dynamic stop condition to be negated for the disconnected case
            stop_condition = lambda r, c: (r <= cY) and not (r == stop_row_r and c == stop_col_r)

        # 2. Execute Tracing Loop
        north_east_count = 0
        south_step = 0
        iters = 0
        
        while stop_condition(row, col):
            # SAFETY CHECKS: Prevent out-of-bounds and infinite loops
            if iters > max_iters or row < 1 or row >= max_row - 1 or col < 1 or col >= max_col - 1:
                break
            iters += 1
            
            if (ven_contour[row-1, col+1] > 0) & (north_east_count <= 10): # north-east
                right_pts.append((col+1, row-1))
                row, col = row - 1, col+1
                north_east_count += 1
                south_step = 0
            elif (ven_contour[row-1, col] > 0) & (south_step == 0): # north
                right_pts.append((col, row-1))
                row, col = row - 1, col
                south_step = 0
            elif (ven_contour[row-1, col-1] > 0): # north-west
                right_pts.append((col-1, row-1))
                row, col = row-1, col-1  
                south_step = 0
            elif (ven_contour[row, col-1] > 0): # west
                right_pts.append((col-1, row))
                row, col = row, col-1   
                south_step = 0
            elif (ven_contour[row+1, col-1] > 0): # south-west
                right_pts.append((col-1, row+1))
                row, col = row+1, col-1
                south_step = 0
            else: # south
                right_pts.append((col, row+1)) 
                row, col = row+1, col  
                south_step = 1

    # ==========================================
    # Unified Post-Processing: Filtering
    # ==========================================
    left_pts = list(set(left_pts))
    right_pts = list(set(right_pts))
    
    left_con_cand_pts = []
    if left_pts:
        try:
            left_min_col, left_max_col = min([x for x, y in left_pts]), max([x for x, y in left_pts])
            left_x_range = left_max_col - left_min_col
            left_low_thresh = left_max_col - 0.2 * left_x_range   
            left_con_cand_pts = [(x, y) for x, y in left_pts if (x <= left_low_thresh)]
        except:
            left_con_cand_pts = left_pts

    right_con_cand_pts = []
    if right_pts:
        try:
            right_max_col, right_min_col = max([x for x, y in right_pts]), min([x for x, y in right_pts])
            right_x_range = right_max_col - right_min_col
            right_low_thresh = right_min_col + 0.2 * right_x_range    
            right_con_cand_pts = [(x, y) for x, y in right_pts if (x >= right_low_thresh)]
        except:
            right_con_cand_pts = right_pts

    return left_con_cand_pts, right_con_cand_pts, left_pts, right_pts

In [ ]:
def append_mirror_pts(lat_ventricles, left_con_cand_pts, right_con_cand_pts):
    """
    Reflects the superior ventricular contour points across their respective centroids.

    This function is designed to bypass segmentation interference caused by the 
    choroid plexus in the lower portions of the lateral ventricles. Rather than 
    tracing the noisy, inferior boundaries, it takes the cleanly traced superior 
    candidate points and mathematically reflects them across the left and right 
    ventricular centroids. This synthesizes a clean, symmetric, and elliptical 
    lower contour.

    Parameters
    ----------
    lat_ventricles : numpy.ndarray
        The binary mask of the segmented lateral ventricles.
    left_con_cand_pts : list of tuples
        The filtered (x, y) candidate points from the superior left ventricle contour.
    right_con_cand_pts : list of tuples
        The filtered (x, y) candidate points from the superior right ventricle contour.

    Returns
    -------
    left_pts_inv : list of tuples
        The (x, y) coordinates representing the synthesized inferior left ventricle contour.
    right_pts_inv : list of tuples
        The (x, y) coordinates representing the synthesized inferior right ventricle contour.
    """
    
    [r, c] = lat_ventricles.nonzero()
    
    if len(c) == 0:
        print("Warning: lat_ventricles mask is empty. Cannot compute mirror points.")
        return [], []
        
    midpoint_c = (min(c) + max(c)) // 2
    
    left_pts_inv = []
    if len(left_con_cand_pts) > 0:
        # Isolate the left half of the mask and calculate its centroid
        left_half = np.uint8(lat_ventricles[:, :midpoint_c])
        lv_M = cv.moments(left_half)
        
        if lv_M["m00"] != 0:
            cX_L = int(lv_M["m10"] / lv_M["m00"])
            cY_L = int(lv_M["m01"] / lv_M["m00"])
            
            # Reflect points across the centroid (simplified from: x - 2*(x - cX))
            left_pts_inv = [(2 * cX_L - x, 2 * cY_L - y) for x, y in left_con_cand_pts]

    right_pts_inv = []  
    if len(right_con_cand_pts) > 0:
        # Isolate the right half of the mask and calculate its centroid
        right_half = np.uint8(lat_ventricles[:, midpoint_c:])
        rv_M = cv.moments(right_half)

        if rv_M["m00"] != 0:
            cX_R = int(rv_M["m10"] / rv_M["m00"]) + midpoint_c 
            cY_R = int(rv_M["m01"] / rv_M["m00"])
            
            # Reflect points across the centroid
            right_pts_inv = [(2 * cX_R - x, 2 * cY_R - y) for x, y in right_con_cand_pts]

    return left_pts_inv, right_pts_inv

In [ ]:
def calculate_max_ecc(coronal_plane, lat_ventricles, left_con_cand_pts, right_con_cand_pts):
    """
    Fits an ellipse to the ventricular contour points and evaluates curvature metrics.

    This function uses a least-squares regression approach to fit an ellipse equation of 
    the form:
    $$A(x-c_x)^2 + B(y-c_y)^2 + C(x-c_x)(y-c_y) = 1$$
    to the provided candidate points for both the left and right ventricles. It extracts 
    the angle of rotation (alpha) and the semi-major/minor axes (a, b). 
    From these, it calculates the major and minor curvatures and the eccentricity. 
    Finally, it plots the coronal plane overlaid with the candidate points.

    Parameters
    ----------
    coronal_plane : numpy.ndarray
        The 2D image array of the current coronal slice for visualization.
    lat_ventricles : numpy.ndarray
        The binary mask of the segmented lateral ventricles used to calculate centroids.
    left_con_cand_pts : list of tuples
        The (x, y) coordinates of the left ventricle contour.
    right_con_cand_pts : list of tuples
        The (x, y) coordinates of the right ventricle contour.

    Returns
    -------
    k_major_L : float
        Major curvature of the fitted left ellipse (returns -100 on failure).
    k_minor_L : float
        Minor curvature of the fitted left ellipse (returns -100 on failure).
    ecc_L : float
        Eccentricity of the fitted left ellipse (returns -100 on failure).
    k_major_R : float
        Major curvature of the fitted right ellipse (returns -100 on failure).
    k_minor_R : float
        Minor curvature of the fitted right ellipse (returns -100 on failure).
    ecc_R : float
        Eccentricity of the fitted right ellipse (returns -100 on failure).
    """
    
    [r,c] = lat_ventricles.nonzero()
    left_plot_flag, right_plot_flag = 1,1

    # Early exit to prevent ValueError if the mask is entirely empty
    if len(c) == 0:
        return -100, -100, -100, -100, -100, -100

    # ==========================================
    # Left Ventricle Processing
    # ==========================================
    if len(left_con_cand_pts) > 0:

        try:
            lv_M = cv.moments(np.uint8(lat_ventricles[:,:(min(c)+max(c))//2]))
            cX_L = int(lv_M["m10"] / lv_M["m00"]) 
            cY_L = int(lv_M["m01"] / lv_M["m00"])
    
            X_L = np.array([((x-cX_L)**2,(y-cY_L)**2,(x-cX_L)*(y-cY_L)) for x,y in left_con_cand_pts]).reshape(len(left_con_cand_pts),3)
            Y_L = np.ones((len(left_con_cand_pts),1))

            coefs_L = np.matmul(np.linalg.inv(np.matmul(np.transpose(X_L),X_L)),np.matmul(np.transpose(X_L),Y_L))
            alpha_L = np.arctan(((coefs_L[2] - coefs_L[0] + coefs_L[1])/(coefs_L[0]-coefs_L[1])) + 1)/2
            b_L = np.sqrt(abs(2/(coefs_L[0]+coefs_L[1]-((coefs_L[2]-coefs_L[0]+coefs_L[1])/((np.tan(2*alpha_L)-1)*(np.cos(2*alpha_L)))))))
            a_L = np.sqrt(abs(1/(coefs_L[0]+coefs_L[1]-1/b_L**2)))

            left_el = np.zeros(lat_ventricles.shape)
            for y in range(left_el.shape[0]):
                for x in range(left_el.shape[1]):
                    f = ((((x-cX_L)*np.cos(alpha_L) + (y-cY_L)*np.sin(alpha_L))/a_L)**2 
                            + (((x-cX_L)*np.sin(alpha_L) - (y-cY_L)*np.cos(alpha_L))/b_L)**2)
                    if f <= 1:
                        left_el[y,x] = 1
            if a_L >= b_L:
                k_major_L, k_minor_L, ecc_L = 1/(b_L**2/a_L)[0], 1/(a_L**2/b_L)[0], np.sqrt(1-(b_L**2)/(a_L**2))[0]
            else:
                k_major_L, k_minor_L, ecc_L = 1/(a_L**2/b_L)[0], 1/(b_L**2/a_L)[0], np.sqrt(1-(a_L**2)/(b_L**2))[0]
        except:
            k_major_L, k_minor_L, ecc_L = -100,-100,-100
            left_plot_flag = 0
    else:
        k_major_L, k_minor_L, ecc_L = -100,-100,-100
        left_plot_flag = 0

    # ==========================================
    # Right Ventricle Processing
    # ==========================================
        
    if len(right_con_cand_pts) > 0:

        try:
            rv_M = cv.moments(np.uint8(lat_ventricles[:,(min(c)+max(c))//2:]))
            cX_R = int(rv_M["m10"] / rv_M["m00"]) + (min(c)+max(c))//2
            cY_R = int(rv_M["m01"] / rv_M["m00"])
    
            X_R = np.array([((x-cX_R)**2,(y-cY_R)**2,(x-cX_R)*(y-cY_R)) for x,y in right_con_cand_pts]).reshape(len(right_con_cand_pts),3)
            Y_R = np.ones((len(right_con_cand_pts),1))


            coefs_R = np.matmul(np.linalg.inv(np.matmul(np.transpose(X_R),X_R)),np.matmul(np.transpose(X_R),Y_R))
            alpha_R = np.arctan(((coefs_R[2] - coefs_R[0] + coefs_R[1])/(coefs_R[0]-coefs_R[1])) + 1)/2
            b_R = np.sqrt(abs(2/(coefs_R[0]+coefs_R[1]-((coefs_R[2]-coefs_R[0]+coefs_R[1])/((np.tan(2*alpha_R)-1)*(np.cos(2*alpha_R)))))))
            a_R = np.sqrt(abs(1/(coefs_R[0]+coefs_R[1]-1/b_R**2)))

            right_el = np.zeros(lat_ventricles.shape)
            for y in range(right_el.shape[0]):
                for x in range(right_el.shape[1]):
                    f = ((((x-cX_R)*np.cos(alpha_R) + (y-cY_R)*np.sin(alpha_R))/a_R)**2 
                          + (((x-cX_R)*np.sin(alpha_R) - (y-cY_R)*np.cos(alpha_R))/b_R)**2 )
                    if f <= 1:
                        right_el[y,x] = 1

            if a_R >= b_R:
                k_major_R, k_minor_R, ecc_R = 1/(b_R**2/a_R)[0], 1/(a_R**2/b_R)[0], np.sqrt(1-(b_R**2)/(a_R**2))[0]
            else:
                k_major_R, k_minor_R, ecc_R = 1/(a_R**2/b_R)[0], 1/(b_R**2/a_R)[0], np.sqrt(1-(a_R**2)/(b_R**2))[0]    
        
        except:
            k_major_R, k_minor_R, ecc_R = -100,-100, -100
            right_plot_flag = 0
    else:
        k_major_R, k_minor_R, ecc_R = -100,-100,-100
        right_plot_flag = 0

    # ==========================================
    # Visualization
    # ==========================================
            
    fig, ax = plt.subplots(figsize = (10,10))
    ax.imshow(coronal_plane, cmap = 'gray')
    if (left_plot_flag == 1):
        [ax.scatter(x,y,marker = 'x',color =   "w", s = 5) for (x,y) in left_con_cand_pts]
    if (right_plot_flag == 1):
        [ax.scatter(x,y,marker = 'x', color = "w",s = 5) for (x,y) in right_con_cand_pts]
    plt.show()
    
    return max(ecc_L,ecc_R)


##### Data setup

###### Before running the pipeline, make a copy of config.template.json, rename it to config.json, and update the paths to point to your local dataset.

In [ ]:
# Load the configuration file
with open('config.json', 'r') as file:
    config = json.load(file)

# Assign variables dynamically
data_path = config['data_path']
coronal_acpc_vol_aligned_name = config['coronal_acpc_vol_aligned_name']
info_csv_path = config['info_csv_path_w_name']
scan_id_col_name = config['scan_id_col_name']
pat_id_col_name = config['pat_id_col_name']
ac_coordinates_col_name = config['ac_coordinates_col_name']
pc_coordinates_col_name = config['pc_coordinates_col_name']

output_folder = config['output_csv_write_folder']

print(f"Loading data from: {data_path}")

In [ ]:
info_df = pd.read_csv(info_csv_path)
info_df[scan_id_col_name] = info_df[scan_id_col_name].str.strip("'")

In [ ]:
accnum_list = info_df[scan_id_col_name].values

In [ ]:
len(accnum_list), len(info_df)

In [ ]:
#dataframe which will contain track of the computed MaxEccLV values for each scan
max_ecc_lv_df = pd.DataFrame()

###### Predefined, emperically determined processing parameters corresponding to ROI sizes, positions, and global/local thresholding parameters. 
The following parameters (ei_calc_processing_params) work for a diverse cohort of chronic neurodegenerative conditions spanning Normal Pressure Hydrocephalus, Alzheimer's disease, post-traumatic encephalomalacia, and headache. So they are recommended to be used directly for patient scans without acute pathology like significant midline shifts, bleeds, or mass effects. 

In [ ]:
coronal_plane_processing_params = dict()
coronal_plane_processing_params["coronal_initial_gauss_blur"] = 3
coronal_plane_processing_params["coronal_close_kernel_size"] = 4
coronal_plane_processing_params["coronal_close_num_iterations"] = 2
coronal_plane_processing_params["coronal_adaptive_thresh_size"] = 75
coronal_plane_processing_params["coronal_adaptive_thresh_mean_thresh"] = 7.5
coronal_plane_processing_params["suppress_label_thresh"] = 0.05

In [ ]:
ventricles_processing_smoothing_params = dict()
ventricles_processing_smoothing_params["middle_roi_upper_dec"] = 5
ventricles_processing_smoothing_params["middle_roi_lower_inc"] = 7
ventricles_processing_smoothing_params["middle_roi_left_dec"] = 20
ventricles_processing_smoothing_params["middle_roi_right_dec"] = 20
ventricles_processing_smoothing_params["scale_upper_thresh"] = 0.5
ventricles_processing_smoothing_params["smoothing_factor"] = 7
ventricles_processing_smoothing_params["dec_thresh"] = 0 
ventricles_processing_smoothing_params["ventricle_to_brain_ratio_thresh"] = 0.015

##### MaxEccLV Pipeline

In [ ]:
for acc_num in accnum_list:

    pat = info_df[pat_id_col_name][(info_df[scan_id_col_name]==acc_num)].values[0] #patient identifier

    pca_str = info_df[pc_coordinates_col_name][(info_df[scan_id_col_name]==acc_num)].values[0]
    [x,y,z] = pca_str.split(",")
    pca_x,pca_y,pca_z = [float(x.strip("[")),float(y),float(z.strip("]"))]            
  
    print(f'Accession number is {acc_num}')
       
    cor_img_path = os.path.join(data_path, acc_num, coronal_acpc_vol_aligned_name)
    pc = [pca_x,pca_y,pca_z]
    
    #Obtain the initially segmented lateral ventricular and full brain masks
    ventricles, full_brain_mask, cor_pc_index, coronal_plane= segment_ventricles(cor_img_path, pc, coronal_plane_processing_params, visualize = False)

    #process the initial segmentation further to obtain a smoothed segmentation of the lateral ventricles
    ventricles_smoothed = process_venticles(ventricles, cor_pc_index, full_brain_mask, ventricles_processing_smoothing_params, visualize = False)

    #determines if the left and right lateral ventricles appear as one joint mass (mostly the case in NPH) or as disconnected blobs
    disconnected_case, lat_ventricles = num_connected_comps_cv(ventricles_smoothed, visualize = False)

    #Needed to guide contour tracing
    M = cv.moments(np.uint8(lat_ventricles))
    if M["m00"] != 0:
        cX = int(M["m10"] / M["m00"])
        cY = int(M["m01"] / M["m00"])
    else:
        cX, cY = lat_ventricles.shape[1]//2, lat_ventricles.shape[0]//2

    
    #kick-off special processing branch if the ventricles appear disconnected
    process_left, process_right = 1, 1         
    if disconnected_case:
        lat_ventricles, disconnected_case, left_ven_con, right_ven_con, process_left, process_right = check_if_disconnected_case_too_small(
                                                                    lat_ventricles, full_brain_mask,
                                                                    disconnected_case, 
                                                                    process_left, process_right, visualize = False)
        left_ven_blobs = measure.label(left_ven_con)
        if len(np.bincount(left_ven_blobs.flatten())) > 2:
            process_left = 0
            
        right_ven_blobs = measure.label(right_ven_con)
        if len(np.bincount(right_ven_blobs.flatten())) > 2:
            process_right = 0
        
    else:
        left_ven_con, right_ven_con = -1, -1
    
    
    #Trace upper contour of the segmented lateral ventricles    
    left_con_cand_pts, right_con_cand_pts, left_pts, right_pts = trace_contour(lat_ventricles, left_ven_con, 
                                                                               right_ven_con,
                                                                               process_left, 
                                                                               process_right,
                                                                               disconnected_case, cX, cY)
        
                
    #Reflect traced upper contour to fit ellipses (this avoids choroid plexus at the bottom)  
    left_pts_inv, right_pts_inv = append_mirror_pts(lat_ventricles,left_con_cand_pts,
                                                                      right_con_cand_pts)

    #Calculate eccentricities of the left and right lateral ventricles
    if len(left_con_cand_pts) > 0:           
        left_con_cand_pts = left_con_cand_pts + left_pts_inv


    if len(right_con_cand_pts) > 0:
        right_con_cand_pts = right_con_cand_pts + right_pts_inv
    

    max_ecc = calculate_max_ecc(coronal_plane,lat_ventricles, left_con_cand_pts, right_con_cand_pts)

    max_ecc_lv_df = pd.concat([max_ecc_lv_df, pd.DataFrame({pat_id_col_name:[pat],scan_id_col_name:["'" + acc_num + "'"],
                                  'max_ecc':[max_ecc]})],
                     ignore_index = True)


In [ ]:
len(max_ecc_lv_df), sum(max_ecc_lv_df['max_ecc'].isna())

In [ ]:
#Examine histogram of computed values
plt.hist(max_ecc_lv_df['max_ecc'])
plt.xticks([x for x in np.arange(0.4, 1, 0.05)])
plt.show()

In [ ]:
#Check for any extreme values and inspect the corresponding outputs above for computational errors
min_max_ecc_lim = 0.4
max_max_ecc_lim = 1

max_ecc_lv_df[(max_ecc_lv_df['max_ecc'] <= min_max_ecc_lim) | (max_ecc_lv_df['max_ecc'] >= max_max_ecc_lim)]

In [ ]:
max_ecc_lv_df.to_csv(os.path.join(output_folder, 'max_ecc_lv.csv'))